# Round 4 v14 Detailed Findings: Mark-Optimized BSM/Binomial Strategy

This notebook documents the current v14 strategy and the research behind it. The goal is to use encrypted trader identities where they help, but avoid the earlier failure mode of manually overfitting to a single Mark or noisy outlier.

v14 combines two layers:

1. **BSM/binomial option pricing base**: a structural model for VEV vouchers using online realized volatility and discrete time-to-expiry.
2. **Optimized Mark overlay**: automatically learned encrypted-trader signals, filtered for sample size, spread-aware profitability, and cross-day consistency.

The Mark overlay is deliberately small. It should tilt trades, not dominate the pricing model.

## 1. Why v14 exists

Earlier versions showed two opposite failure modes:

- Pure Mark-following could overfit individual counterparties and produce negative drift when the signal did not appear or reversed.
- Pure option pricing was more defensible, but left potential alpha on the table when the data clearly contained repeated encrypted-trader behavior.

v14 tries to sit between those extremes. It searches for useful Marks systematically, then uses them only as a weak overlay on top of the BSM/binomial fair value framework.

## 2. Data loaded

The research uses the Round 4 price and trade files:

- `prices_round_4_day_1.csv`
- `prices_round_4_day_2.csv`
- `prices_round_4_day_3.csv`
- `trades_round_4_day_1.csv`
- `trades_round_4_day_2.csv`
- `trades_round_4_day_3.csv`

The price data is used for forward mark-to-market checks and option model backtests. The trade data is used to scan encrypted counterparty behavior.

In [ ]:
import pandas as pd
from IPython.display import Image, display

product_summary = pd.read_csv('/mnt/data/round4_mark_optimized_v14_detailed/product_summary.csv')
mark_summary = pd.read_csv('/mnt/data/round4_mark_optimized_v14_detailed/mark_activity_summary.csv')
signals = pd.read_csv('/mnt/data/round4_mark_optimized_v14_detailed/optimized_signal_explanation.csv')
all_edges = pd.read_csv('/mnt/data/round4_mark_optimized_v14/all_mark_edge_scan.csv')
pnl = pd.read_csv('/mnt/data/round4_mark_optimized_v14/v14_bsm_backtest_pnl_by_product.csv')
product_summary

## 3. Spread matters

The scan is **spread-aware**. This is critical because many apparent Mark signals look good if you compare trade time to future mid, but disappear once you assume you actually need to cross the spread.

For example:

- To go long after a Mark buys, the simulated entry is at the current ask.
- To go short after a Mark sells, the simulated entry is at the current bid.

This avoids treating non-executable mid-price moves as real edge.

In [ ]:
display(Image('/mnt/data/round4_mark_optimized_v14_detailed/median_spread_by_product.png'))

## 4. Mark scan methodology

For every product, Mark, buyer/seller role, and forward horizon, the scan evaluates two possible actions:

### Long after event

If a Mark appears and we choose to go long:

\[
\text{PnL}_{long} = \text{future mid} - \text{current ask}
\]

### Short after event

If a Mark appears and we choose to go short:

\[
\text{PnL}_{short} = \text{current bid} - \text{future mid}
\]

The scan tests horizons such as 1, 2, 5, 10, 20, 50, 100, and 200 rows ahead. This lets the model identify whether a Mark's edge is immediate microstructure flow or slower directional information.

## 5. Filters to reduce overfitting

A Mark signal is retained only if it passes several filters:

- At least **60 total quantity**.
- At least **5 events**.
- Positive average PnL per unit **after spread**.
- Positive on at least **2 out of 3 days** where data exists.
- Sample-size shrinkage is applied so tiny samples cannot dominate.
- Only the top few signals per product are kept.

This is meant to prevent the strategy from blindly trusting one outlier, especially the type of Mark-specific mean reversion behavior that harmed earlier versions.

In [ ]:
signals.sort_values('score', ascending=False).head(30)

In [ ]:
display(Image('/mnt/data/round4_mark_optimized_v14_detailed/detailed_optimized_mark_scores.png'))
display(Image('/mnt/data/round4_mark_optimized_v14_detailed/retained_signal_avg_pnl.png'))
display(Image('/mnt/data/round4_mark_optimized_v14_detailed/signal_score_by_product.png'))

## 6. Why not just follow the best Mark?

The all-edge distribution below shows why following Marks blindly is dangerous. Most Mark/product/role/horizon combinations are not useful after spread. The retained signals are a small subset of all possible signals.

This is the key reason v14 uses Marks as an overlay rather than the main engine.

In [ ]:
display(Image('/mnt/data/round4_mark_optimized_v14_detailed/mark_edge_distribution.png'))

## 7. BSM/binomial base model

The main valuation layer remains the v13 option model.

For each VEV voucher, the strategy computes:

1. Black-Scholes call value.
2. CRR binomial call value using the discrete time grid.
3. A blended fair value.
4. A disagreement penalty when BSM and binomial prices differ.

A voucher is traded only when bid/ask mispricing exceeds:

\[
\max(\text{edge floor}, 0.25 \cdot \text{spread}, 0.02 \cdot |\text{fair}|, |\text{BSM} - \text{binomial}|)
\]

This keeps the strategy from trading small noisy model discrepancies.

## 8. Discrete binomial enforcement

The binomial model uses the explicit risk-neutral terminal distribution:

\[
C = \sum_{j=0}^{N} \binom{N}{j} p^j(1-p)^{N-j}\max(Su^jd^{N-j}-K,0)
\]

where:

\[
u = e^{\sigma\sqrt{\Delta t}}, \quad d = \frac{1}{u}, \quad p = \frac{1-d}{u-d}
\]

This enforces a discrete distribution over future VEV prices rather than relying only on continuous BSM assumptions. It also helps reduce overfit to short-lived Mark behavior.

## 9. How Marks enter v14

In v14, the learned Mark signal does **not** replace fair value.

For vouchers:

- Mark flow shifts the fair value modestly.
- Mark flow can create a small position tilt.
- But large directional trades still require BSM/binomial edge.

For VEV:

- Mark flow combines with the VEV trend score.

For Hydrogel:

- Only optimized Mark signals are used.
- There is no manually designed Mark 14 mean-reversion rule.

## 10. BSM/binomial base backtest

The uploaded activities log does not include enough public market-trade identity information to perfectly replay the Mark overlay. Therefore, the table below shows the BSM/binomial base component backtest. The Mark overlay is assessed using the historical data scan.

In [ ]:
pnl.sort_values('estimated_pnl', ascending=False)

In [ ]:
display(Image('/mnt/data/round4_mark_optimized_v14_detailed/detailed_bsm_pnl_by_product.png'))
display(Image('/mnt/data/round4_mark_optimized_v14_detailed/detailed_bsm_equity_path.png'))

Estimated BSM/binomial base PnL on the uploaded activity log: **11,563**.

The Mark overlay is designed to improve entries when retained Mark signals appear live, but it is intentionally constrained so that it cannot completely override the pricing model.

## 11. Current strategy strengths

- Much less path-specific than terminal anchor strategies.
- Less Mark-overfit than hand-selected counterparty rules.
- Uses executable bid/ask edge, not midpoint fantasy edge.
- Combines structural option pricing with empirical flow information.
- Marks are auto-scanned and filtered rather than manually chosen.

This makes v14 a stronger compromise between raw profitability and defensibility.

## 12. Remaining caveats

- The Mark overlay is still learned offline, so it may not generalize if the hidden trader behavior changes.
- The BSM/binomial model depends on the volatility estimate and time-to-expiry assumption.
- The activity-log backtest cannot fully replay Mark overlay performance because the live log format does not provide the exact same public market trade identity stream for simulation.
- If maximizing score is the only objective and the path is fixed, path-specific methods can outperform this. But v14 is more defensible and less brittle.

## 13. Final recommendation

Use v14 when you want a strategy that is:

- not terminal-price hardcoded,
- not manually Mark-overfit,
- option-theoretically grounded,
- still able to exploit encrypted trader patterns when they are statistically supported.

If the next run is still negative, the next diagnostic should compare live trade fills against the retained Mark list to see whether the overlay is helping or hurting in real time.